# Face Authentication - Testing / Inference
Run `task2_training.ipynb` first to generate `model_config.json` before using this.

In [1]:
import numpy as np
import cv2
import json
from deepface import DeepFace

In [2]:
def load_model_config(config_path="model_config.json"):
    with open(config_path, "r") as f:
        config = json.load(f)
    print(f"Config loaded: model={config['model_name']}, threshold={config['threshold']}")
    return config

In [3]:
def get_face_bounding_boxes(img_path, detector_backend):
    faces = DeepFace.extract_faces(
        img_path=img_path,
        detector_backend=detector_backend,
        enforce_detection=False
    )
    boxes = []
    for face in faces:
        region = face.get("facial_area", {})
        if region:
            boxes.append({
                "x": region.get("x"),
                "y": region.get("y"),
                "w": region.get("w"),
                "h": region.get("h")
            })
    return boxes

In [4]:
def predict(img1_path, img2_path, config):
    model_name = config["model_name"]
    detector = config["detector_backend"]
    threshold = config["threshold"]

    rep1 = DeepFace.represent(
        img_path=img1_path,
        model_name=model_name,
        detector_backend=detector,
        enforce_detection=False
    )
    rep2 = DeepFace.represent(
        img_path=img2_path,
        model_name=model_name,
        detector_backend=detector,
        enforce_detection=False
    )

    emb1 = np.array(rep1[0]["embedding"])
    emb2 = np.array(rep2[0]["embedding"])

    dot = np.dot(emb1, emb2)
    norm = np.linalg.norm(emb1) * np.linalg.norm(emb2)
    score = float(dot / norm) if norm != 0 else 0.0

    result = "same person" if score >= threshold else "different person"

    boxes_img1 = get_face_bounding_boxes(img1_path, detector)
    boxes_img2 = get_face_bounding_boxes(img2_path, detector)

    output = {
        "verification_result": result,
        "similarity_score": round(score, 4),
        "bounding_boxes": {
            "image_1": boxes_img1,
            "image_2": boxes_img2
        }
    }
    return output

In [5]:
config = load_model_config("model_config.json")

Config loaded: model=Facenet, threshold=0.7


In [6]:
image1 = "sample_images/person1_a.jpg"
image2 = "sample_images/person1_b.jpg"

output = predict(image1, image2, config)

print("\n--- Result ---")
print(f"Verification : {output['verification_result']}")
print(f"Similarity   : {output['similarity_score']}")
print(f"Bounding Box Image 1 : {output['bounding_boxes']['image_1']}")
print(f"Bounding Box Image 2 : {output['bounding_boxes']['image_2']}")


--- Result ---
Verification : different person
Similarity   : 0.6784
Bounding Box Image 1 : [{'x': 90, 'y': 111, 'w': 240, 'h': 240}]
Bounding Box Image 2 : [{'x': 120, 'y': 121, 'w': 274, 'h': 274}]
